# Inclusive prefix sum (research prototype)

Compute the running total of a sequence: `out[i] = in[0] + in[1] + ... + in[i]`.
The input uses a fixed deterministic fill so the result is reproducible; we report
the checksum (sum of all `out` values) as the oracle. The C port must implement the
inclusive scan and match this checksum.

In [ ]:
N_TEST = 1000000

# deterministic pseudo-random fill in [-0.5, 0.5); exact integer arithmetic, so a C
# port using unsigned 64-bit arithmetic reproduces every value bit-for-bit
def in_of(i):
    return ((i * 2654435761) % 1000003) / 1000003.0 - 0.5

xs = [in_of(i) for i in range(N_TEST)]

In [ ]:
# naive inclusive scan: a single running total (inherently sequential as written)
def scan(xs):
    out = []
    run = 0.0
    for x in xs:
        run += x
        out.append(run)
    return out

out = scan(xs)
checksum = 0.0
for v in out:
    checksum += v
print('checksum =', checksum)   # -> 3178464.2888227087

In [ ]:
# The oracle: this notebook's own output. A port of the algorithm (in any
# language) must reproduce this checksum within tol. The out values are handed
# over as the terms of the checksum, so the extraction tool can measure how much
# the sum legitimately moves under reordering. (That spread only covers the final
# reduction -- a parallel block scan also reorders the additions inside each
# prefix -- so it is a lower bound on how much correct ports may disagree, and
# tol must stay above it.)
ORACLE = {'kernel': 'scan',
          'params': {'N': N_TEST},
          'expected': {'CHECKSUM': checksum},
          'terms': {'CHECKSUM': out},
          'tol': 1e-4}